# Low-Light Image Denoising + 4x Super-Resolution

**Task:** Given low-resolution, noisy, low-light images, produce denoised images at **4x** the
input resolution (joint denoising + super-resolution).

**Approach:** Fine-tune a **pretrained Real-ESRGAN (RRDBNet, x4plus)** model.
Real-ESRGAN is trained with a synthetic degradation pipeline that already includes
noise + blur + downsampling, which is a very close match to this task, so a pretrained
checkpoint gives a strong starting point instead of training a CNN/GAN from scratch. We
fine-tune it briefly on the competition's train pairs so it adapts to the actual noise
distribution, validate with PSNR on the val set, then run inference on the test set.

> **Before you run:** this notebook assumes each of `train/`, `val/`, `test/` is a single
> folder containing paired files that can be matched by filename, with the noisy/low-res
> image and the clean/high-res image distinguished by a substring in the filename (e.g.
> `low`/`high`, `LR`/`HR`, `noisy`/`clean`). **Check the `FILENAME PATTERNS` cell below
> against your actual filenames and adjust the two substrings** — this is the one thing
> most likely to need a tweak since the exact naming wasn't specified in the brief.

**Workflow — smoke test first, then full run:** section 2 has a `SAMPLE_MODE` flag.
Leave it `True` for your first **Run All** — it slices the data to a handful of items and
runs 2 epochs, so the whole pipeline (pairing → dataset → model → training → checkpoint →
tiled inference → submission file → HF upload) is exercised end-to-end in a couple of
minutes. Once you see `=== SMOKE TEST PASSED ===`, set `SAMPLE_MODE = False` and **Run
All** again for the real training run on the full dataset.


## 1. Setup — install dependencies

In [1]:
!pip install -q basicsr realesrgan facexlib gfpgan --no-deps
!pip install -q huggingface_hub scikit-image opencv-python-headless


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.5 MB/s eta 0:00:00


In [2]:
import sys, types
import torchvision.transforms.functional as _TF

if 'torchvision.transforms.functional_tensor' not in sys.modules:
    _shim = types.ModuleType('torchvision.transforms.functional_tensor')
    _shim.rgb_to_grayscale = _TF.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _shim
    print('Registered torchvision.transforms.functional_tensor compatibility shim.')

Registered torchvision.transforms.functional_tensor compatibility shim.


`basicsr`/`realesrgan` are installed with `--no-deps` because their pinned dependency
versions are old and otherwise clash with Kaggle's default torch/numpy — we already have
compatible torch, opencv, numpy, Pillow on Kaggle images.


In [3]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import glob, random, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as sk_psnr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


Device: cuda


## 2. Paths & filename pattern config — **check these against the actual data**

### Sample-mode smoke test
Set **`SAMPLE_MODE = True`** the first time you run this notebook. It slices the data
down to a handful of pairs/files and runs only 1-2 epochs, so the *entire* pipeline —
pairing, dataset, model load, training loop, checkpointing, tiled inference, submission
file, HF upload cell — executes end-to-end in a couple of minutes. Once that finishes
without errors, flip it to **`SAMPLE_MODE = False`** and do **Run All** again for the
real training run on the full dataset. This avoids discovering a bug (bad path, shape
mismatch, OOM on a big test image) only after an hour-long full training run.


In [4]:
# --- SAMPLE MODE TOGGLE ---
SAMPLE_MODE = False     # True: quick end-to-end smoke test on a small subset.
                       # False: full run on all train/val/test data.

SAMPLE_TRAIN_N = 12    # number of train pairs used when SAMPLE_MODE=True
SAMPLE_VAL_N   = 4     # number of val pairs used when SAMPLE_MODE=True
SAMPLE_TEST_N  = 4     # number of test files used when SAMPLE_MODE=True
SAMPLE_EPOCHS  = 2     # epochs used when SAMPLE_MODE=True
FULL_EPOCHS    = 8     # epochs used when SAMPLE_MODE=False

EPOCHS = SAMPLE_EPOCHS if SAMPLE_MODE else FULL_EPOCHS

if SAMPLE_MODE:
    print('*** SAMPLE_MODE = True — quick smoke test on a small subset. ***')
    print('*** Once this run completes cleanly end-to-end, set SAMPLE_MODE = False and Run All again. ***')
else:
    print('*** SAMPLE_MODE = False — full run on the complete dataset. ***')


*** SAMPLE_MODE = False — full run on the complete dataset. ***


In [5]:
# Kaggle typically mounts competition data under /kaggle/input/<competition-slug>/
DATA_ROOT = '/kaggle/input/low-light-sr'                      # adjust if your comp folder differs
COMP_DIR  = glob.glob(f'{DATA_ROOT}/*')      # try to auto-locate; fall back to manual
if not COMP_DIR:
    COMP_DIR = glob.glob(f'{DATA_ROOT}/*')
COMP_DIR = COMP_DIR[0] if COMP_DIR else DATA_ROOT
print('Using competition dir:', COMP_DIR)

TRAIN_DIR = "/kaggle/input/low-light-sr/train"
VAL_DIR   = "/kaggle/input/low-light-sr/val"
TEST_DIR  = "/kaggle/input/low-light-sr/test"


Using competition dir: /kaggle/input/low-light-sr/val


### Quick look at the actual folder structure

Run this before trusting any auto-pairing logic — it tells you which strategy below will
actually apply (subfolders vs. one flat folder), and shows a couple of real filenames and
image sizes so you can sanity-check the automatic pairing once it runs.


In [6]:
def inspect_folder(folder, n=6):
    print(f'--- {folder} ---')
    if not os.path.isdir(folder):
        print('  (does not exist)')
        return
    entries = sorted(os.listdir(folder))
    subdirs = [e for e in entries if os.path.isdir(os.path.join(folder, e))]
    files   = [e for e in entries if os.path.isfile(os.path.join(folder, e))]
    print(f'  subfolders: {subdirs if subdirs else "(none)"}')
    print(f'  files here: {len(files)} (showing up to {n}): {files[:n]}')
    for f in files[:3]:
        img = cv2.imread(os.path.join(folder, f))
        if img is not None:
            print(f'    {f} -> shape {img.shape}')

for d in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    inspect_folder(d)


--- /kaggle/input/low-light-sr/train ---
  subfolders: ['gt', 'train']
  files here: 0 (showing up to 6): []
--- /kaggle/input/low-light-sr/val ---
  subfolders: ['gt', 'val']
  files here: 0 (showing up to 6): []
--- /kaggle/input/low-light-sr/test ---
  subfolders: (none)
  files here: 60 (showing up to 6): ['test_00001.png', 'test_00002.png', 'test_00003.png', 'test_00004.png', 'test_00005.png', 'test_00006.png']
    test_00001.png -> shape (160, 256, 3)
    test_00002.png -> shape (160, 256, 3)
    test_00003.png -> shape (160, 256, 3)


### Pairing strategy — matches your actual layout: `train/gt/` + `train/train/`

Your data has, inside `train/`, a **`gt`** subfolder (ground-truth clean/high-res images)
and a same-named **`train`** subfolder (low-res noisy inputs) — `val/` is presumably laid
out the same way (`val/gt/` + `val/val/`), and `test/` likely has just the input images
(no `gt`, since test has no ground truth). The pairing logic below handles this, in order:

1. **Two subfolders present** — if one is named `gt` (or `ground_truth`/`hr`/`high`/
   `clean`/`target`), that one is used as the high-res folder unambiguously and the other
   subfolder (whatever it's named — here, `train`) is the low-res one. If neither
   subfolder matches a known "high-res" name, it falls back to guessing by average
   resolution. Either way, files across the two subfolders are then paired by filename
   stem, or by numeric ID if stems don't match exactly (e.g. `0001.png` / `img_0001.png`).
2. **Filename tag** (`LOW_TAG`/`HIGH_TAG`), only used if there are no subfolders.
3. **Flat single-folder fallback**: groups files by resolution, then pairs by numeric ID.
4. Sorted-order pairing as a last resort, with a loud warning — verify `train_pairs[0]`.

For `test/`: if there are no files directly inside `test/` but exactly one subfolder
(e.g. `test/test/`), it automatically descends into that subfolder.


In [7]:
import re

# --- FILENAME PATTERNS --- only used as a fallback strategy; safe to leave as-is
LOW_TAG  = 'low'    # substring identifying the noisy / low-res image, e.g. 'low', 'LR', 'noisy'
HIGH_TAG = 'high'   # substring identifying the clean / high-res image, e.g. 'high', 'HR', 'clean'

def _immediate_subdirs(folder):
    return sorted(os.path.join(folder, d) for d in os.listdir(folder)
                  if os.path.isdir(os.path.join(folder, d)))

def _avg_pixel_area(folder, sample=5):
    files = sorted(glob.glob(os.path.join(folder, '*')))[:sample]
    areas = [cv2.imread(f).shape[0] * cv2.imread(f).shape[1]
             for f in files if cv2.imread(f) is not None]
    return np.mean(areas) if areas else 0

def _extract_id(fname):
    """Numeric ID embedded in a filename, e.g. 'img_0001_gt.png' -> '0001'."""
    digits = re.findall(r'\d+', os.path.basename(fname))
    return digits[0] if digits else None

def _pair_by_stem_or_id(lows, highs):
    # exact stem match first
    high_by_stem = {os.path.splitext(os.path.basename(f))[0]: f for f in highs}
    pairs = [(lf, high_by_stem[os.path.splitext(os.path.basename(lf))[0]])
             for lf in lows if os.path.splitext(os.path.basename(lf))[0] in high_by_stem]
    if pairs:
        return pairs, 'filename stem'
    # numeric-ID match next
    high_by_id = {}
    for f in highs:
        i = _extract_id(f)
        if i is not None:
            high_by_id[i] = f
    pairs = []
    for lf in lows:
        i = _extract_id(lf)
        if i is not None and i in high_by_id:
            pairs.append((lf, high_by_id[i]))
    if pairs:
        return pairs, 'numeric ID'
    # last resort: sorted order, only if counts match
    if len(lows) == len(highs) and len(lows) > 0:
        return list(zip(sorted(lows), sorted(highs))), 'SORTED ORDER (unverified guess!)'
    return [], 'none'

def list_pairs(folder):
    subdirs = _immediate_subdirs(folder)
    if len(subdirs) == 2:
        a, b = subdirs
        # Name-based detection first (e.g. 'gt' == ground truth == high-res clean) —
        # this is a stronger, unambiguous signal, so prefer it over resolution guessing.
        HIGH_NAMES = {'gt', 'ground_truth', 'groundtruth', 'hr', 'high', 'clean', 'target'}
        name_a, name_b = os.path.basename(a).lower(), os.path.basename(b).lower()
        if name_a in HIGH_NAMES and name_b not in HIGH_NAMES:
            high_dir, low_dir = a, b
        elif name_b in HIGH_NAMES and name_a not in HIGH_NAMES:
            high_dir, low_dir = b, a
        else:
            low_dir, high_dir = (a, b) if _avg_pixel_area(a) <= _avg_pixel_area(b) else (b, a)
        print(f'[{folder}] two subfolders detected -> low: {os.path.basename(low_dir)}, '
              f'high (gt): {os.path.basename(high_dir)}')
        lows  = sorted(glob.glob(os.path.join(low_dir, '*')))
        highs = sorted(glob.glob(os.path.join(high_dir, '*')))
        pairs, how = _pair_by_stem_or_id(lows, highs)
        print(f'[{folder}] paired {len(pairs)} items via {how}')
        return pairs

    all_files = sorted(f for f in glob.glob(os.path.join(folder, '*')) if os.path.isfile(f))
    lows  = [f for f in all_files if LOW_TAG.lower()  in os.path.basename(f).lower()]
    highs = [f for f in all_files if HIGH_TAG.lower() in os.path.basename(f).lower()]
    if lows and highs:
        pairs, how = _pair_by_stem_or_id(lows, highs)
        if pairs:
            print(f'[{folder}] paired {len(pairs)} items via filename tag + {how}')
            return pairs

    # single flat folder, no subdirs, no tags -> split by resolution
    print(f'[{folder}] no subfolders or tags found; grouping by image resolution instead.')
    sized = []
    for f in all_files:
        img = cv2.imread(f)
        if img is not None:
            sized.append((f, img.shape[0] * img.shape[1], img.shape[:2]))
    if not sized:
        return []
    sized.sort(key=lambda x: x[1])
    median_area = sized[len(sized) // 2][1]
    low_group  = [f for f, a, _ in sized if a <= median_area]
    high_group = [f for f, a, _ in sized if a > median_area]
    print(f'[{folder}] resolution split -> {len(low_group)} low-res, {len(high_group)} '
          f'high-res  (e.g. low {sized[0][2]}, high {sized[-1][2]} — confirm high is ~4x low)')
    pairs, how = _pair_by_stem_or_id(low_group, high_group)
    print(f'[{folder}] paired {len(pairs)} items via {how}')
    return pairs

def list_test_files(folder):
    """Test set has no ground truth, so just the noisy/low-res files. If the folder
    itself has no files but exactly one subfolder (mirroring the train/val 'gt' + input
    layout, minus 'gt'), descend into that subfolder."""
    direct = sorted(f for f in glob.glob(os.path.join(folder, '*')) if os.path.isfile(f))
    if direct:
        return direct
    subdirs = _immediate_subdirs(folder)
    if len(subdirs) == 1:
        print(f'[{folder}] no files directly here; using subfolder {os.path.basename(subdirs[0])}')
        return sorted(f for f in glob.glob(os.path.join(subdirs[0], '*')) if os.path.isfile(f))
    return direct

train_pairs = list_pairs(TRAIN_DIR)
val_pairs   = list_pairs(VAL_DIR)
test_files  = list_test_files(TEST_DIR)

print('full train pairs:', len(train_pairs))
print('full val pairs:  ', len(val_pairs))
print('full test files: ', len(test_files))
if train_pairs:
    print('example pair:', train_pairs[0])
    lo = cv2.imread(train_pairs[0][0]); hi = cv2.imread(train_pairs[0][1])
    print('  low shape:', lo.shape, '| high shape:', hi.shape,
          '(high should be ~4x low in each dimension)')

if SAMPLE_MODE:
    rng = random.Random(SEED)
    train_pairs = rng.sample(train_pairs, min(SAMPLE_TRAIN_N, len(train_pairs)))
    val_pairs   = rng.sample(val_pairs,   min(SAMPLE_VAL_N, len(val_pairs)))
    test_files  = rng.sample(test_files,  min(SAMPLE_TEST_N, len(test_files)))
    print(f'[SAMPLE_MODE] using {len(train_pairs)} train / {len(val_pairs)} val / '
          f'{len(test_files)} test for the smoke test')


[/kaggle/input/low-light-sr/train] two subfolders detected -> low: train, high (gt): gt
[/kaggle/input/low-light-sr/train] paired 1105 items via filename stem
[/kaggle/input/low-light-sr/val] two subfolders detected -> low: val, high (gt): gt
[/kaggle/input/low-light-sr/val] paired 267 items via numeric ID
full train pairs: 1105
full val pairs:   267
full test files:  60
example pair: ('/kaggle/input/low-light-sr/train/train/gt_00001.png', '/kaggle/input/low-light-sr/train/gt/gt_00001.png')
  low shape: (160, 256, 3) | high shape: (640, 1024, 3) (high should be ~4x low in each dimension)


## 3. Dataset

In [8]:
class DenoiseSRDataset(Dataset):
    """Returns (LR noisy image, HR clean image) as float tensors in [0,1].
    Random crop + flip augmentation for training; center image (resized-safe) for val."""
    def __init__(self, pairs, patch_size=128, scale=4, train=True):
        self.pairs = pairs
        self.patch_size = patch_size
        self.scale = scale
        self.train = train

    def __len__(self):
        return len(self.pairs)

    def _read(self, path):
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return img

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        lr = self._read(lr_path)
        hr = self._read(hr_path)

        ps = self.patch_size
        s = self.scale
        h, w, _ = lr.shape
        # guard against images smaller than the patch size
        ps_h = min(ps, h)
        ps_w = min(ps, w)

        if self.train:
            top  = random.randint(0, h - ps_h)
            left = random.randint(0, w - ps_w)
        else:
            top  = (h - ps_h) // 2
            left = (w - ps_w) // 2

        lr_patch = lr[top:top+ps_h, left:left+ps_w, :]
        hr_patch = hr[top*s:(top+ps_h)*s, left*s:(left+ps_w)*s, :]

        if self.train and random.random() < 0.5:
            lr_patch = np.ascontiguousarray(lr_patch[:, ::-1, :])
            hr_patch = np.ascontiguousarray(hr_patch[:, ::-1, :])

        lr_t = torch.from_numpy(lr_patch.transpose(2, 0, 1)).float()
        hr_t = torch.from_numpy(hr_patch.transpose(2, 0, 1)).float()
        return lr_t, hr_t

train_ds = DenoiseSRDataset(train_pairs, patch_size=64, scale=4, train=True)
val_ds   = DenoiseSRDataset(val_pairs,   patch_size=64, scale=4, train=False)

BATCH_SIZE = 2          # lowered from 8 to fit a 16GB GPU at this patch size
ACCUM_STEPS = 4         # gradient accumulation -> effective batch size 2*4 = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2)

print('train batches:', len(train_loader), '| val batches:', len(val_loader))
print(f'effective batch size: {BATCH_SIZE} x {ACCUM_STEPS} accum steps = {BATCH_SIZE*ACCUM_STEPS}')


train batches: 552 | val batches: 267
effective batch size: 2 x 4 accum steps = 8


## 4. Model — pretrained Real-ESRGAN (RRDBNet, x4plus)

We download the official `RealESRGAN_x4plus.pth` weights and load them into the
`RRDBNet` architecture from `basicsr`. This checkpoint is already trained for the
denoise + 4x-SR combination, so it's a strong initialization to fine-tune from.

**Compatibility shim:** newer `torchvision` versions removed the internal
`torchvision.transforms.functional_tensor` module, which older `basicsr` releases still
import (for a function that now lives in `torchvision.transforms.functional` under the
same name). The cell below registers a shim module *before* importing `basicsr` so that
import succeeds without needing to patch the installed package or downgrade torchvision.


In [9]:
import sys, types
import torchvision.transforms.functional as _TF

if 'torchvision.transforms.functional_tensor' not in sys.modules:
    _shim = types.ModuleType('torchvision.transforms.functional_tensor')
    _shim.rgb_to_grayscale = _TF.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _shim
    print('Registered torchvision.transforms.functional_tensor compatibility shim.')


In [10]:
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.utils.download_util import load_file_from_url

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)

WEIGHTS_URL = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
weights_path = load_file_from_url(WEIGHTS_URL, model_dir='weights', progress=True)

state_dict = torch.load(weights_path, map_location='cpu')
state_dict = state_dict.get('params_ema', state_dict.get('params', state_dict))
model.load_state_dict(state_dict, strict=True)
model = model.to(device)
print('Loaded pretrained RealESRGAN_x4plus weights.')


Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /kaggle/working/weights/RealESRGAN_x4plus.pth



100%|██████████| 63.9M/63.9M [00:00<00:00, 203MB/s]


Loaded pretrained RealESRGAN_x4plus weights.


## 5. Fine-tune on the competition's train pairs

In [11]:
# EPOCHS was set in section 2 (SAMPLE_EPOCHS / FULL_EPOCHS based on SAMPLE_MODE)
LR = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.99))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
l1_loss = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

def evaluate_psnr(loader):
    model.eval()
    psnrs = []
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                sr = model(lr).clamp(0, 1)
            sr_np = sr.squeeze(0).permute(1, 2, 0).float().cpu().numpy()
            hr_np = hr.squeeze(0).permute(1, 2, 0).float().cpu().numpy()
            psnrs.append(sk_psnr(hr_np, sr_np, data_range=1.0))
            del sr, lr, hr
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    model.train()
    return float(np.mean(psnrs)) if psnrs else float('nan')

best_psnr = -1
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (lr_img, hr_img) in enumerate(train_loader):
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            sr = model(lr_img)
            loss = l1_loss(sr, hr_img) / ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUM_STEPS
        del sr, lr_img, hr_img, loss
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    scheduler.step()

    val_psnr = evaluate_psnr(val_loader) if len(val_loader) > 0 else float('nan')
    print(f'Epoch {epoch:02d}/{EPOCHS} | train L1 {running_loss/max(1,len(train_loader)):.4f} '
          f'| val PSNR {val_psnr:.3f} dB | {time.time()-t0:.1f}s')

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({'params_ema': model.state_dict()}, 'checkpoints/best_model.pth')
        print(f'  -> new best (PSNR {best_psnr:.3f}), checkpoint saved')

print('Best val PSNR:', best_psnr)


Epoch 01/8 | train L1 0.0111 | val PSNR 38.691 dB | 121.2s
  -> new best (PSNR 38.691), checkpoint saved
Epoch 02/8 | train L1 0.0096 | val PSNR 38.866 dB | 121.6s
  -> new best (PSNR 38.866), checkpoint saved
Epoch 03/8 | train L1 0.0097 | val PSNR 38.779 dB | 118.4s
Epoch 04/8 | train L1 0.0094 | val PSNR 38.937 dB | 120.4s
  -> new best (PSNR 38.937), checkpoint saved
Epoch 05/8 | train L1 0.0095 | val PSNR 39.003 dB | 119.4s
  -> new best (PSNR 39.003), checkpoint saved
Epoch 06/8 | train L1 0.0094 | val PSNR 38.983 dB | 119.2s
Epoch 07/8 | train L1 0.0093 | val PSNR 39.028 dB | 118.2s
  -> new best (PSNR 39.028), checkpoint saved
Epoch 08/8 | train L1 0.0092 | val PSNR 39.040 dB | 118.7s
  -> new best (PSNR 39.040), checkpoint saved
Best val PSNR: 39.04020343637591


## 6. Load best checkpoint and run inference on the test set

Always check your own `val_psnr` before submitting.


In [12]:
ckpt = torch.load('checkpoints/best_model.pth', map_location=device)
model.load_state_dict(ckpt['params_ema'])
model.eval()

os.makedirs('outputs/test_sr', exist_ok=True)

def infer_image(path, tile=256, tile_pad=16, scale=4):
    """Tiled inference to keep memory bounded on large test images."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    h, w, _ = img.shape
    t = torch.from_numpy(img.transpose(2, 0, 1)).float().unsqueeze(0).to(device)

    if max(h, w) <= tile:
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            out = model(t).clamp(0, 1)
    else:
        out = torch.zeros(1, 3, h * scale, w * scale, device=device)
        for y in range(0, h, tile):
            for x in range(0, w, tile):
                y0, y1 = max(0, y - tile_pad), min(h, y + tile + tile_pad)
                x0, x1 = max(0, x - tile_pad), min(w, x + tile + tile_pad)
                patch = t[:, :, y0:y1, x0:x1]
                with torch.no_grad(), torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    sr_patch = model(patch).clamp(0, 1)
                oy0, oy1 = (y - y0) * scale, (y - y0) * scale + min(tile, h - y) * scale
                ox0, ox1 = (x - x0) * scale, (x - x0) * scale + min(tile, w - x) * scale
                out[:, :, y*scale:y*scale+(oy1-oy0), x*scale:x*scale+(ox1-ox0)] = \
                    sr_patch[:, :, oy0:oy1, ox0:ox1]

    out_np = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255.0).round().clip(0, 255).astype(np.uint8)
    return out_np

pred_records = []
for f in test_files:
    out_img = infer_image(f)
    fname = os.path.basename(f)
    out_path = os.path.join('outputs/test_sr', fname)
    Image.fromarray(out_img).save(out_path)
    pred_records.append(fname)

print(f'Wrote {len(pred_records)} super-resolved images to outputs/test_sr/')

if SAMPLE_MODE:
    assert len(pred_records) > 0, 'Smoke test produced no outputs — check paths/tags above.'
    print()
    print('=== SMOKE TEST PASSED ===')
    print('Data loading, pairing, model load, fine-tuning, checkpointing, tiled inference,')
    print('and file writing all completed without errors on the small subset.')
    print('Next: set SAMPLE_MODE = False in section 2, then Run All for the full run.')


Wrote 60 super-resolved images to outputs/test_sr/


## 7. Build `submission.csv`

The brief provides a `submission.py` script for the correct submission format, which
wasn't included in the uploaded instructions — **run the competition's own
`submission.py` against `outputs/test_sr/` to generate the exact `submission.csv`**
(column layout for image-to-value competitions varies). The cell below is a generic
placeholder (filename + path) so you have something to adapt if `submission.py` isn't
available in your environment.


In [13]:
import importlib.util, glob as _glob

_candidates = _glob.glob('/kaggle/input/**/submission.py', recursive=True)
if not _candidates:
    raise FileNotFoundError(...)
SUBMISSION_SCRIPT_PATH = _candidates[0]

with open(SUBMISSION_SCRIPT_PATH, 'r') as f:
    _script_src = f.read()

# strip the script's own hardcoded bottom call (uses placeholder 'path_to_images')
_cutoff = _script_src.find("folder_path = 'path_to_images'")
if _cutoff != -1:
    _script_src = _script_src[:_cutoff]

_exec_globals = {}
exec(_script_src, _exec_globals)
images_to_csv = _exec_globals['images_to_csv']

images_to_csv(folder_path='outputs/test_sr', output_csv='submission.csv')

Successfully saved to submission.csv


In [14]:
import pandas as pd
data=pd.read_csv("submission.csv")
data.head


<bound method NDFrame.head of           ID  pixel_0  pixel_1  pixel_2  pixel_3  pixel_4  pixel_5  pixel_6  \
0   gt_00004       61       62       64       64       67       69       80   
1   gt_00006       37       51       50       33       30       31       30   
2   gt_00039       39       73       95       72       62       61       81   
3   gt_00051      101      106      107      108      106      109      110   
4   gt_00043       38       32       29       33       39       36       37   
5   gt_00019       55       56       54       53       52       44       48   
6   gt_00038       50       35       40       44       40       59       40   
7   gt_00009       29       33       34       28       28       38       29   
8   gt_00024       39       49       50       54       53       50       49   
9   gt_00059       21       21       66       82       79       80       80   
10  gt_00042       48       44       45       40       44       47       44   
11  gt_00020       61 

## 8. Push the fine-tuned model to Hugging Face Hub

Optionally save the fine-tuned model to a Hugging Face repo. Fill in `HF_TOKEN` (as a Kaggle secret, not hardcoded) and
`REPO_ID` (e.g. `your-username/low-light-denoise-sr`).


In [15]:
from huggingface_hub import HfApi, create_repo

HF_TOKEN = None   # set via Kaggle "Add-ons > Secrets", do not hardcode your token
REPO_ID  = 'your-username/low-light-denoise-sr'   # <-- change this

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True)
    api.upload_file(
        path_or_fileobj='checkpoints/best_model.pth',
        path_in_repo='best_model.pth',
        repo_id=REPO_ID,
        token=HF_TOKEN,
    )
    print(f'Uploaded checkpoint to https://huggingface.co/{REPO_ID}')
else:
    print('Set HF_TOKEN (and REPO_ID) above, then re-run this cell to push the model.')


Set HF_TOKEN (and REPO_ID) above, then re-run this cell to push the model.


## Notes / things to double-check before submitting
- **Workflow:** run once with `SAMPLE_MODE = True` (fast smoke test on ~12 train / 4 val /
  4 test items, 2 epochs) to catch path/pairing/shape/OOM issues cheaply, then flip to
  `SAMPLE_MODE = False` and **Run All** again for the real training run.
- **`LOW_TAG` / `HIGH_TAG`** in section 2 — the actual filename convention wasn't given, so
  verify pairing worked (`train_pairs[0]` should show a genuinely matching low/high pair).
- **`patch_size`** (64) and **`FULL_EPOCHS`** (8) are conservative defaults for a short
  training budget — increase both if you have GPU budget left, since more fine-tuning
  usually keeps improving PSNR on this type of task.
